> ⚠️ **DO NOT run the code in this notebook directly!** 
> This script was used with the full DTU dataset downloaded.
>
> 💡 **If you want to run it:**
> 1. Download the dataset from [SGD Repository DTU Zenodo](https://doi.org/10.5281/zenodo.8202150).
> 2. Point the path in the code to your local directory: `./your_machine_path/sgd_data/square_runs`.

## Data Source
I downloaded the complete dataset from the project repository, which includes:

- The original code used to generate all test cases

- The simulation results

- Approximately 10.68 GB of data covering multiple wind farm configurations

The key folder used in this step, in my computer is: sgd_data/square_runs


Inside this folder, the data is structured hierarchically based on the simulation parameters:

1. **Optimization Method:** `det` (deterministic SLSQP) or `sgd`
2. **Turbine Layout Density:** Organized by the number of rows (`rows_10`, `rows_12`, `rows_15`, `rows_18`)
3. **Random Seed:** Directory blocks ranging from `seed_1` to `seed_20`

---

###  Visual Directory Structure

```text
[method]/
└── [rows]/
    └── [seed]/
        ├── [method]_x.npy  <- Array storing turbine X coordinates
        └── [method]_y.npy  <- Array storing turbine Y coordinates


The Python script below automates the extraction process:

In [4]:
#!/usr/bin/env python3
import numpy as np
from pathlib import Path

base_path = Path("/Users/brunoboer/Downloads/sgd_data/square_runs") #this is in my local machine
output = Path.cwd().parent / "Cases_DTU" # you can check this folder here : Optimizers_Bruno/Designs/Cases_DTU
output.mkdir(exist_ok=True)

row_configs = {
    10: (100, (10-1)*5*80),
    12: (144, (12-1)*5*80),
    15: (225, (15-1)*5*80),
    18: (324, (18-1)*5*80)
}

for rows_str, (n_turb, L) in row_configs.items():
    det_dir = base_path / f"rows_{rows_str}_det"
    
    if not det_dir.exists():
        print(f"{det_dir} does not exist")
        continue
    
    pasta_rows = output / f"rows_{rows_str}_T"
    pasta_rows.mkdir(exist_ok=True)
    
    x0, y0, x1, y1 = 0.0, 0.0, L, L
    print(f"\nProcessing {det_dir.name} -> {n_turb} turbines [{L:.0f}m]")
    
    seeds_ok = 0
    for seed in range(1, 21):
        seed_dir = det_dir / f"seed_{seed}"  
        
        if not (seed_dir / 'det_x.npy').exists():
            continue  
        
        sgd_x = np.load(seed_dir / 'det_x.npy')
        sgd_y = np.load(seed_dir / 'det_y.npy')
        x_init, y_init = sgd_x[0], sgd_y[0]
        
        if len(x_init) == n_turb:
            csv_name = f"{n_turb}turb_{L:.0f}m_kdt_{seed}.csv"
            csv_path = pasta_rows / csv_name
            
            with open(csv_path, 'w') as f:
                f.write("x_coordinate,y_coordinate,x0,y0,x1,y1\n")
                for xi, yi in zip(x_init, y_init):
                    f.write(f"{xi:.6f},{yi:.6f},{x0:.1f},{y0:.1f},{x1:.1f},{y1:.1f}\n")
            
            seeds_ok += 1
            print(f"  seed_{seed} -> {csv_name} saved")
    
    print(f"  Summary: {seeds_ok}/20 completed -> {pasta_rows.name}")

print(f"\nCases_DTU finished ")


Processing rows_10_det -> 100 turbines [3600m]
  seed_1 -> 100turb_3600m_kdt_1.csv saved
  seed_2 -> 100turb_3600m_kdt_2.csv saved
  seed_3 -> 100turb_3600m_kdt_3.csv saved
  seed_4 -> 100turb_3600m_kdt_4.csv saved
  seed_5 -> 100turb_3600m_kdt_5.csv saved
  seed_6 -> 100turb_3600m_kdt_6.csv saved
  seed_7 -> 100turb_3600m_kdt_7.csv saved
  seed_8 -> 100turb_3600m_kdt_8.csv saved
  seed_9 -> 100turb_3600m_kdt_9.csv saved
  seed_10 -> 100turb_3600m_kdt_10.csv saved
  seed_11 -> 100turb_3600m_kdt_11.csv saved
  seed_12 -> 100turb_3600m_kdt_12.csv saved
  seed_13 -> 100turb_3600m_kdt_13.csv saved
  seed_14 -> 100turb_3600m_kdt_14.csv saved
  seed_15 -> 100turb_3600m_kdt_15.csv saved
  seed_16 -> 100turb_3600m_kdt_16.csv saved
  seed_17 -> 100turb_3600m_kdt_17.csv saved
  seed_18 -> 100turb_3600m_kdt_18.csv saved
  seed_19 -> 100turb_3600m_kdt_19.csv saved
  seed_20 -> 100turb_3600m_kdt_20.csv saved
  Summary: 20/20 completed -> rows_10_T

Processing rows_12_det -> 144 turbines [4400m]
  

### Results

Now, this layouts files will contains the mapping between row configurations and turbine counts:

```
rows_10 → 100 turbines, domain size L = 3600m

rows_12 → 144 turbines, domain size L = 4000m

rows_15 → 225 turbines, domain size L = 4800m

rows_18 → 324 turbines, domain size L = 5600m
```

So then it saves each valid initial design as a CSV file with the format:



| x_coordinate | y_coordinate | x0 | y0 | x1 | y1 |
|---|---|---|---|---|---|
| xxxx | xxxx | 0 | 0 | 3600 | 3600 |


Where x0, y0, x1, y1 define the domain boundaries.



### How is organized?

So, the directory tree is now organized as follows:
```
Cases_DTU/
    rows_10_T/
        100turb_3600m_kdt_1.csv
        100turb_3600m_kdt_2.csv
        ...
    rows_12_T/
        144turb_4000m_kdt_1.csv
        ...
    rows_15_T/
        ...
    rows_18_T/
        ...
```

These CSV files will be used as the initial designs for both my optimizer and SGD